# Hey Emma — Custom Wake Word Training

Trainiert ein openWakeWord-Modell für **"Hey Emma"** mit synthetischen Stimmen (Piper TTS).

**Voraussetzungen:** Google Colab mit GPU-Runtime (T4 reicht, beschleunigt TTS-Generierung).

**Ergebnis:** `hey_emma.onnx` — bereit für den Einsatz in der Hey Emma App.

## 1. Environment Setup

In [ ]:
# Install openWakeWord from GitHub (PyPI package doesn't include train.py)
!pip install git+https://github.com/dscripka/openWakeWord.git

# Training dependencies (not auto-installed, torchaudio is pre-installed on Colab)
!pip install onnx onnxruntime datasets huggingface_hub pyyaml \
    torchinfo torchmetrics mutagen audiomentations pronouncing \
    speechbrain torch-audiomentations acoustics

In [ ]:
# Clone piper-sample-generator (script-based tool, not a pip package)
!git clone https://github.com/dscripka/piper-sample-generator.git

# Install its dependencies
!pip install -r piper-sample-generator/requirements.txt

# Download the Piper TTS model (LibriTTS, ~904 speakers, ~1 GB)
!mkdir -p piper-sample-generator/models
!wget -q -O piper-sample-generator/models/en-us-libritts-high.pt \
    'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'

import os
PIPER_PATH = os.path.abspath('piper-sample-generator')
print(f'Piper path: {PIPER_PATH}')
print(f'Model exists: {os.path.exists(os.path.join(PIPER_PATH, "models", "en-us-libritts-high.pt"))}')

## 2. Download Training Data

- **Negative features:** ~2000h ACAV100M (vorberechnete Audio-Embeddings)
- **Validation set:** ~11h für False-Positive-Rate-Messung
- **Room Impulse Responses:** MIT RIRs für Augmentierung
- **Background noise:** Umgebungsgeräusche für Augmentierung

In [ ]:
import os
os.makedirs('training_data', exist_ok=True)
os.chdir('training_data')

In [ ]:
# Download pre-computed negative features from HuggingFace (~2000 hrs, ~16 GB)
from huggingface_hub import hf_hub_download

negative_features_path = hf_hub_download(
    repo_id='davidscripka/openwakeword_features',
    filename='openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    repo_type='dataset',
    local_dir='.',
)
print(f'Negative features: {negative_features_path}')

In [ ]:
# Download false-positive validation set
validation_path = hf_hub_download(
    repo_id='davidscripka/openwakeword_features',
    filename='validation_set_features.npy',
    repo_type='dataset',
    local_dir='.',
)
print(f'Validation features: {validation_path}')

In [ ]:
# Download MIT Room Impulse Responses
!wget -q https://www.openslr.org/resources/28/rirs_noises.zip
!unzip -q -o rirs_noises.zip -d mit_rirs
!rm rirs_noises.zip
print('MIT RIRs downloaded.')

In [ ]:
# Download background noise dataset (MUSAN)
!wget -q https://www.openslr.org/resources/17/musan.tar.gz
!tar -xzf musan.tar.gz
!rm musan.tar.gz
print('Background noise (MUSAN) downloaded.')

In [ ]:
os.chdir('..')
print(f'Working directory: {os.getcwd()}')

## 3. Training Config

Die Config wird direkt hier erstellt — keine externe Datei nötig.

In [ ]:
import yaml, os

PIPER_PATH = '/content/piper-sample-generator'

config = {
    'model_name': 'hey_emma',
    'target_phrase': ['hey emma'],
    'custom_negative_phrases': [
        'hey anna', 'hey ella', 'hey eva', 'hey ever',
        'hey oma', 'hey irma', 'hey mama', 'hey lemma', 'hey thema',
    ],
    'n_samples': 50000,
    'n_samples_val': 5000,
    'tts_batch_size': 100,
    'piper_sample_generator_path': PIPER_PATH,
    'output_dir': './hey_emma_model',
    'augmentation_batch_size': 16,
    'augmentation_rounds': 2,
    'rir_paths': ['./training_data/mit_rirs/RIRS_NOISES/simulated_rirs'],
    'background_paths': ['./training_data/musan/noise'],
    'background_paths_duplication_rate': [1],
    'feature_data_files': {
        'ACAV100M_sample': './training_data/openwakeword_features_ACAV100M_2000_hrs_16bit.npy',
    },
    'false_positive_validation_data_path': './training_data/validation_set_features.npy',
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('hey_emma.yml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print('Config written to hey_emma.yml')

## 4. Write Training Wrapper

Compatibility shims for Colab (torchaudio API change + PyTorch 2.6 weights_only default).

In [ ]:
%%writefile run_train.py
# --- Compatibility shims for Colab ---

# 1. torchaudio >= 2.2 removed list_audio_backends()
import torchaudio
if not hasattr(torchaudio, 'list_audio_backends'):
    torchaudio.list_audio_backends = lambda: ['soundfile', 'sox']

# 2. PyTorch 2.6 changed torch.load default to weights_only=True
#    Piper's model needs the old behavior (trusted local model file)
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# 3. Add piper-sample-generator to sys.path so train.py can import it
import sys
sys.path.insert(0, '/content/piper-sample-generator')

# --- Run training script ---
import openwakeword, os

TRAIN_SCRIPT = os.path.join(os.path.dirname(openwakeword.__file__), 'train.py')
print(f'train.py: {TRAIN_SCRIPT}')

phase = sys.argv[1] if len(sys.argv) > 1 else '--generate_clips'
sys.argv = [TRAIN_SCRIPT, '--training_config', 'hey_emma.yml', phase]
exec(open(TRAIN_SCRIPT).read())

## 5. Generate Synthetic Clips

Piper TTS erzeugt 50.000 Varianten von "Hey Emma" mit ~904 verschiedenen Stimmen.

In [ ]:
!python run_train.py --generate_clips

## 6. Augment Clips

Wendet Room Impulse Responses und Hintergrundgeräusche an.
2 Augmentation-Runden verdoppeln die Datenvielfalt.

In [ ]:
!python run_train.py --augment_clips

## 7. Train Model

Trainiert ein kleines DNN (2x 32 Units) auf den Audio-Embeddings.
Ziel: ≤ 0.2 False Positives pro Stunde.

In [ ]:
!python run_train.py --train_model

## 8. Export to ONNX + TFLite

In [ ]:
!python run_train.py --convert_to_tflite

In [ ]:
import glob, os

models = glob.glob('hey_emma_model/**/*.onnx', recursive=True)
models += glob.glob('hey_emma_model/**/*.tflite', recursive=True)

print('Exportierte Modelle:')
for m in sorted(models):
    size_kb = os.path.getsize(m) / 1024
    print(f'  {m} ({size_kb:.0f} KB)')

## 9. Quick Test

Schnelltest mit synthetischem Audio, um zu prüfen ob das Modell reagiert.

In [ ]:
import numpy as np, glob, wave
from openwakeword.model import Model

onnx_models = glob.glob('hey_emma_model/**/*.onnx', recursive=True)
model_path = [m for m in onnx_models if 'hey_emma' in m and 'embedding' not in m and 'melspec' not in m]

if model_path:
    print(f'Testing model: {model_path[0]}')
    oww = Model(wakeword_models=[model_path[0]], inference_framework='onnx')
    print(f'Loaded models: {list(oww.models.keys())}')

    val_clips = glob.glob('hey_emma_model/**/positive_val/*.wav', recursive=True)
    if val_clips:
        with wave.open(val_clips[0], 'rb') as wf:
            audio = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16)

        max_score = 0.0
        for i in range(0, len(audio) - 1280, 1280):
            chunk = audio[i:i+1280]
            prediction = oww.predict(chunk)
            for name, score in prediction.items():
                max_score = max(max_score, score)

        print(f'Max score on positive clip: {max_score:.3f}')
        print('PASS' if max_score > 0.5 else 'WARN: score below threshold, consider retraining')
    else:
        print('No validation clips found for testing.')
else:
    print('ERROR: No ONNX model found in output directory.')

## 10. Download Model

Das fertige Modell herunterladen und in `resources/` des Hey Emma Projekts ablegen.

In `.env` setzen:
```
OPENWAKEWORD_MODEL_PATH=hey_emma.onnx
OPENWAKEWORD_KEYWORD=Hey-Emma
```

In [ ]:
try:
    from google.colab import files
    if model_path:
        files.download(model_path[0])
        print('Download gestartet.')
except ImportError:
    if model_path:
        print(f'Modell bereit: {model_path[0]}')
        print('Kopiere die Datei nach resources/hey_emma.onnx im Projekt.')